# 13d · Fixing the OOF→LB Gap (0.87 OOF → 0.61 LB)

## Diagnosis of 12d failure

| Symptom | Cause | Fix |
|---|---|---|
| OOF 0.87, LB 0.61 → gap of 0.26 | Model memorised train folds | Stronger regularisation + augmentation |
| 47% test images confidence < 0.6 | Features don't generalise to test memes | Label smoothing + mixup |
| Fold 1 OOF 0.91 vs others ~0.85 | High variance across folds | More stable training (lower LR, more patience) |

## Changes in 13d
- **EfficientNetB3** (stronger features, 300×300 native res)
- **Much stronger augmentation**: random rotation ±30°, shear, coarse dropout
- **Mixup** (α=0.3): blends pairs of images + labels → forces smooth decision boundary
- **Label smoothing** (ε=0.1): prevents overconfident wrong predictions
- **Unfreeze gradually**: top-20 layers first, then full network
- **Monitor val balanced accuracy** directly, not val accuracy
- **Cosine decay with warm restarts** for fine-tune phase
- **Conservative TTA**: only horizontal flip (proven, not noisy)


## 0 · Install & Imports

In [6]:
%pip install -q tensorflow scikit-learn pandas pillow tqdm matplotlib



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import os, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB3
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print('TF:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))


TF: 2.21.0
GPUs: []


## 1 · Config

In [8]:
BASE_DIR = '.'
IMG_DIR  = os.path.join(BASE_DIR, 'img')

IMG_SIZE     = 300      # EfficientNetB3 native resolution
IMG_CHANNELS = 3

# Training
BATCH_SIZE   = 16       # smaller batch → more gradient noise → better generalisation
EPOCHS_HEAD  = 15       # longer head training
EPOCHS_FINE1 = 15       # unfreeze top layers only
EPOCHS_FINE2 = 20       # unfreeze full network

LR_HEAD  = 5e-4
LR_FINE1 = 2e-5         # top layers
LR_FINE2 = 5e-6         # full network (very conservative)

# Regularisation
LABEL_SMOOTH = 0.1      # prevent overconfident predictions
MIXUP_ALPHA  = 0.3      # mixup interpolation strength
DROPOUT_RATE = 0.5

N_FOLDS  = 5
UNFREEZE_TOP_N = 30     # layers to unfreeze in fine-tune phase 1

LABEL_MAP = {0: 'Animal Crossing', 1: 'Doom'}


## 2 · Load Labels & Images

In [9]:
train_df = pd.read_csv(os.path.join(BASE_DIR, 'train-label.csv'))
test_df  = pd.read_csv(os.path.join(BASE_DIR, 'test-label.csv'))
print('Train:', train_df.shape, '| Test:', test_df.shape)

def load_image_array(filepath, size=IMG_SIZE):
    img = Image.open(filepath).convert('RGB')
    img = img.resize((size, size), Image.BILINEAR)
    return np.array(img, dtype=np.float32)

def load_all(df, img_dir, has_labels=True):
    X, y = [], []
    missing = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Loading'):
        path = os.path.join(img_dir, row['file'])
        if not os.path.exists(path):
            missing += 1; continue
        X.append(load_image_array(path))
        if has_labels:
            y.append(int(row['label']))
    if missing:
        print(f'  ⚠️  {missing} images not found')
    X = np.stack(X).astype(np.float32)
    y = np.array(y, dtype=np.int32) if has_labels else None
    return X, y

print('Loading train ...')
X_all, y_all = load_all(train_df, IMG_DIR, has_labels=True)
print(f'  {X_all.shape}')

print('Loading test ...')
X_test, _ = load_all(test_df, IMG_DIR, has_labels=False)
print(f'  {X_test.shape}')


Train: (1385, 3) | Test: (1386, 3)
Loading train ...


Loading: 100%|██████████| 1385/1385 [01:06<00:00, 20.74it/s]


  (1385, 300, 300, 3)
Loading test ...


Loading: 100%|██████████| 1386/1386 [01:04<00:00, 21.39it/s]


  (1386, 300, 300, 3)


## 3 · Augmentation + Mixup

**Mixup:** for each batch, randomly blend two images and their labels:  
`x_mix = λ·x_i + (1-λ)·x_j`, `y_mix = λ·y_i + (1-λ)·y_j` where λ ~ Beta(α,α)

This forces the model to produce smooth, interpolated outputs between classes — directly attacking overconfident wrong predictions.


In [10]:
def augment_image(image):
    """Strong augmentation applied per-image (tf.function compatible)."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.25)
    image = tf.image.random_contrast(image, 0.6, 1.4)
    image = tf.image.random_saturation(image, 0.6, 1.4)
    image = tf.image.random_hue(image, 0.08)
    # Random crop: pad 12% then crop back
    pad = int(IMG_SIZE * 0.12)
    image = tf.image.pad_to_bounding_box(
        image, pad, pad, IMG_SIZE + 2*pad, IMG_SIZE + 2*pad)
    image = tf.image.random_crop(image, [IMG_SIZE, IMG_SIZE, IMG_CHANNELS])
    # Coarse dropout: zero out random 20x20 patch
    h = tf.random.uniform([], 0, IMG_SIZE - 20, dtype=tf.int32)
    w = tf.random.uniform([], 0, IMG_SIZE - 20, dtype=tf.int32)
    mask = tf.ones([IMG_SIZE, IMG_SIZE, IMG_CHANNELS])
    patch = tf.zeros([20, 20, IMG_CHANNELS])
    mask = tf.tensor_scatter_nd_update(
        mask,
        [[i, j, c] for i in range(20) for j in range(20) for c in range(3)],
        tf.reshape(patch, [-1])
    )
    # simpler version: just clip
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image

def mixup_batch(images, labels_onehot, alpha=MIXUP_ALPHA):
    #Apply mixup to a batch. labels must be one-hot float
    batch_size = tf.shape(images)[0]
    lam = tf.cast(
        tf.random.Generator.from_seed(SEED).make_uniform([batch_size], 0, 1),
        tf.float32)
    # sample lambda from Beta(alpha, alpha) via a simple approximation
    lam = tf.maximum(lam * alpha + (1 - alpha), 1 - alpha)
    lam = tf.reshape(lam, [-1, 1, 1, 1])
    
    idx = tf.random.shuffle(tf.range(batch_size))
    images2 = tf.gather(images, idx)
    labels2 = tf.gather(labels_onehot, idx)
    
    mixed_images = lam * images + (1 - lam) * images2
    lam_label = tf.reshape(lam, [-1, 1])
    mixed_labels = lam_label * labels_onehot + (1 - lam_label) * labels2
    return mixed_images, mixed_labels

def make_train_ds(X, y, batch_size=BATCH_SIZE, seed=SEED, use_mixup=True):
    AUTOTUNE = tf.data.AUTOTUNE
    num_classes = 2
    
    @tf.function
    def aug_fn(image, label):
        image = augment_image(image)
        return image, label
    
    ds = (tf.data.Dataset
          .from_tensor_slices((X, y))
          .shuffle(len(X), seed=seed)
          .map(aug_fn, num_parallel_calls=AUTOTUNE)
          .batch(batch_size))
    
    if use_mixup:
        def apply_mixup(images, labels):
            labels_oh = tf.one_hot(labels, num_classes)
            return mixup_batch(images, labels_oh)
        ds = ds.map(apply_mixup, num_parallel_calls=AUTOTUNE)
    
    return ds.prefetch(AUTOTUNE)

def make_eval_ds(X, y=None, batch_size=BATCH_SIZE):
    AUTOTUNE = tf.data.AUTOTUNE
    if y is not None:
        ds = tf.data.Dataset.from_tensor_slices((X, y))
    else:
        ds = tf.data.Dataset.from_tensor_slices(X)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

print('Augmentation pipeline ready.')


Augmentation pipeline ready.


## 4 · Custom Balanced Accuracy Callback

In [11]:
class BalancedAccuracyCallback(keras.callbacks.Callback):
    """Computes val balanced accuracy at each epoch — used for early stopping."""
    def __init__(self, val_ds, val_labels):
        super().__init__()
        self.val_ds = val_ds
        self.val_labels = val_labels
        self.best_bal_acc = 0.0
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        probs = self.model.predict(self.val_ds, verbose=0)
        preds = np.argmax(probs, axis=1)
        bal_acc = balanced_accuracy_score(self.val_labels, preds)
        logs['val_bal_acc'] = bal_acc
        if bal_acc > self.best_bal_acc:
            self.best_bal_acc = bal_acc
            self.best_weights = self.model.get_weights()
        if (epoch + 1) % 5 == 0:
            print(f'    Epoch {epoch+1}: val_bal_acc={bal_acc:.4f} (best={self.best_bal_acc:.4f})')


## 5 · Model: EfficientNetB3 with Gradual Unfreezing

In [12]:
def build_model(trainable_backbone=False, unfreeze_top_n=0):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, IMG_CHANNELS))
    
    backbone = EfficientNetB3(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs,
        pooling=None
    )
    
    # Freeze/unfreeze strategy
    if not trainable_backbone:
        backbone.trainable = False
    else:
        backbone.trainable = True
        if unfreeze_top_n > 0:
            # Freeze all except last unfreeze_top_n layers
            for layer in backbone.layers[:-unfreeze_top_n]:
                layer.trainable = False
    
    x = backbone.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(512, activation='relu', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(2, activation='softmax')(x)
    
    return keras.Model(inputs, outputs)

# Loss with label smoothing — key for preventing overconfidence
def smoothed_loss(y_true, y_pred):
    # y_true may be one-hot (mixup) or integer
    if len(y_true.shape) == 1:
        y_true = tf.one_hot(tf.cast(y_true, tf.int32), 2)
    return keras.losses.categorical_crossentropy(
        y_true, y_pred, label_smoothing=LABEL_SMOOTH)

print('Model builder ready.')
m = build_model()
print(f'Total params: {m.count_params():,}')
del m


Model builder ready.
43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Total params: 11,711,537


## 6 · 5-Fold CV — Three-Phase Training

In [13]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_probs  = np.zeros((len(X_all), 2), dtype=np.float32)
test_probs = np.zeros((len(X_test), 2), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_all, y_all)):
    print(f'\n{"="*55}')
    print(f'  FOLD {fold+1}/{N_FOLDS}')
    print(f'{"="*55}')
    
    X_tr, X_val = X_all[tr_idx], X_all[val_idx]
    y_tr, y_val = y_all[tr_idx], y_all[val_idx]
    
    train_ds = make_train_ds(X_tr, y_tr, use_mixup=True)
    val_ds   = make_eval_ds(X_val, y_val)
    test_ds  = make_eval_ds(X_test)
    
    bal_cb = BalancedAccuracyCallback(val_ds, y_val)
    
    # ── Phase 1: Head only ────────────────────────────────────
    print('  Phase 1: head only ...')
    model = build_model(trainable_backbone=False)
    model.compile(
        optimizer=keras.optimizers.Adam(LR_HEAD),
        loss=smoothed_loss, metrics=['accuracy'])
    
    model.fit(train_ds, epochs=EPOCHS_HEAD, validation_data=val_ds,
              callbacks=[bal_cb], verbose=0)
    model.set_weights(bal_cb.best_weights)
    print(f'    Best val_bal_acc: {bal_cb.best_bal_acc:.4f}')
    
    # ── Phase 2: Unfreeze top N layers ────────────────────────
    print(f'  Phase 2: unfreeze top {UNFREEZE_TOP_N} layers ...')
    bal_cb2 = BalancedAccuracyCallback(val_ds, y_val)
    bal_cb2.best_bal_acc = bal_cb.best_bal_acc
    bal_cb2.best_weights = bal_cb.best_weights
    
    model2 = build_model(trainable_backbone=True, unfreeze_top_n=UNFREEZE_TOP_N)
    model2.set_weights(model.get_weights())
    
    steps = len(X_tr) // BATCH_SIZE * EPOCHS_FINE1
    lr2 = keras.optimizers.schedules.CosineDecay(LR_FINE1, steps, alpha=1e-7)
    model2.compile(optimizer=keras.optimizers.Adam(lr2),
                   loss=smoothed_loss, metrics=['accuracy'])
    
    model2.fit(train_ds, epochs=EPOCHS_FINE1, validation_data=val_ds,
               callbacks=[bal_cb2], verbose=0)
    model2.set_weights(bal_cb2.best_weights)
    print(f'    Best val_bal_acc: {bal_cb2.best_bal_acc:.4f}')
    del model
    
    # ── Phase 3: Full fine-tune ───────────────────────────────
    print('  Phase 3: full fine-tune ...')
    bal_cb3 = BalancedAccuracyCallback(val_ds, y_val)
    bal_cb3.best_bal_acc = bal_cb2.best_bal_acc
    bal_cb3.best_weights = bal_cb2.best_weights
    
    model3 = build_model(trainable_backbone=True, unfreeze_top_n=0)
    model3.set_weights(model2.get_weights())
    
    steps3 = len(X_tr) // BATCH_SIZE * EPOCHS_FINE2
    lr3 = keras.optimizers.schedules.CosineDecay(LR_FINE2, steps3, alpha=1e-8)
    model3.compile(optimizer=keras.optimizers.Adam(lr3),
                   loss=smoothed_loss, metrics=['accuracy'])
    
    model3.fit(train_ds, epochs=EPOCHS_FINE2, validation_data=val_ds,
               callbacks=[bal_cb3], verbose=0)
    model3.set_weights(bal_cb3.best_weights)
    print(f'    Best val_bal_acc: {bal_cb3.best_bal_acc:.4f}')
    del model2
    
    # ── OOF ──────────────────────────────────────────────────
    oof_probs[val_idx] = model3.predict(make_eval_ds(X_val), verbose=0)
    fold_ba = balanced_accuracy_score(y_val, np.argmax(oof_probs[val_idx], axis=1))
    fold_scores.append(fold_ba)
    print(f'  → Fold {fold+1} OOF Balanced Accuracy: {fold_ba:.4f}')
    
    # ── TTA test: clean + hflip only (conservative) ──────────
    raw   = model3.predict(test_ds, verbose=0)
    hflip = model3.predict(
        make_eval_ds(X_test[:, :, ::-1, :]), verbose=0)  # horizontal flip
    test_probs += (raw + hflip) / 2 / N_FOLDS
    
    del model3
    keras.backend.clear_session()

print(f'\n{"="*55}')
for i, s in enumerate(fold_scores):
    print(f'  Fold {i+1}: {s:.4f}')
print(f'  Mean OOF BA: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')



  FOLD 1/5


AttributeError: in user code:

    File "/var/folders/w7/h8ptyfn56xnfdf1y_gqj5ssw0000gn/T/ipykernel_10013/222909111.py", line 64, in apply_mixup  *
        return mixup_batch(images, labels_oh)
    File "/var/folders/w7/h8ptyfn56xnfdf1y_gqj5ssw0000gn/T/ipykernel_10013/222909111.py", line 30, in mixup_batch  *
        lam = tf.cast(

    AttributeError: 'Generator' object has no attribute 'make_uniform'


## 7 · Results

In [ ]:
oof_preds = np.argmax(oof_probs, axis=1)
overall_ba = balanced_accuracy_score(y_all, oof_preds)
print(f'Overall OOF Balanced Accuracy: {overall_ba:.4f}')
for cls, name in [(0, 'Animal Crossing'), (1, 'Doom')]:
    mask = y_all == cls
    print(f'  {name}: {(oof_preds[mask]==y_all[mask]).mean():.4f}')

confidence = np.max(test_probs, axis=1)
print(f'\nMean test confidence: {confidence.mean():.3f}')
print(f'Low-confidence (<0.6): {(confidence < 0.6).sum()} / {len(confidence)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar([f'Fold {i+1}' for i in range(N_FOLDS)], fold_scores,
            color='#0652DD', edgecolor='black', alpha=0.8)
axes[0].axhline(np.mean(fold_scores), color='red', linestyle='--',
                label=f'Mean={np.mean(fold_scores):.4f}')
axes[0].set_ylim(0, 1.05); axes[0].legend()
axes[0].set_title('OOF Balanced Accuracy per Fold')

axes[1].hist(confidence, bins=40, color='#0652DD', alpha=0.8)
axes[1].axvline(0.6, color='red', linestyle='--', label='0.6')
axes[1].set_title('Test Confidence Distribution'); axes[1].legend()
plt.tight_layout(); plt.show()


## 8 · Save Submission

In [ ]:
test_preds = np.argmax(test_probs, axis=1)
submission = pd.DataFrame({'id': test_df['id'].values, 'label': test_preds})
submission.to_csv('submission14.csv', index=False)

print(submission['label'].value_counts())
print(f'Saved submission13.csv — {len(submission)} rows')
assert len(submission) == len(test_df)
assert set(submission['label'].unique()).issubset({0,1})
print('✅ Format OK')


## 9 · What to Try if LB is Still Low

If OOF is good but LB still drops:
- The memes in test may be from a different subreddit era — try **test-time domain adaptation**
- Try **ViT-B16** (Vision Transformer) — better at understanding image+text mixtures like memes
- **OCR features**: extract text from memes and use a text classifier alongside CNN — Doom memes have very different text than Animal Crossing memes
- Reduce mixup alpha to 0.1 if OOF–LB gap persists (mixup may be too aggressive)
